In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

# Data load karo
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Preprocessor load karo
preprocessor = joblib.load('../models/preprocessor.pkl')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("Ready!")

X_train shape: (5634, 22)
X_test shape: (1409, 22)
Ready!


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_val_score


models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('prep', preprocessor),
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('model', XGBClassifier(random_state=42, eval_metric='logloss'))
    ]),
    'KNN': Pipeline([
        ('prep', preprocessor),
        ('model', KNeighborsClassifier())
    ])
}

print(f"{len(models)} models ready!")

5 models ready!


In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, 
                             cv=cv, scoring='roc_auc')
    results[name] = scores
    print(f"{name}: AUC = {scores.mean():.3f} (+/- {scores.std():.3f})")

Logistic Regression: AUC = 0.846 (+/- 0.012)
Decision Tree: AUC = 0.660 (+/- 0.012)
Random Forest: AUC = 0.826 (+/- 0.013)
XGBoost: AUC = 0.823 (+/- 0.008)
KNN: AUC = 0.781 (+/- 0.009)


In [4]:
best_model = models['Logistic Regression']
best_model.fit(X_train, y_train)

import joblib
joblib.dump(best_model, '../models/best_model.pkl')

from sklearn.metrics import classification_report
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

